# SPARQL Engine Benchmark

This notebook compares query execution performance across:

- **RDFLib** — local in-memory RDF graph
- **Apache Jena Fuseki** — materialized RDF graph exposed as a SPARQL endpoint
- **Ontop** — virtual RDF graph queried through a SPARQL endpoint


## Versions

- **RDFLib 7.6.0**  
  https://rdflib.readthedocs.io/en/stable/  
  Install with: `pip install rdflib==7.6.0`

- **Ontop 5.5.0**  
  https://github.com/ontop/ontop/releases/download/ontop-5.5.0/ontop-cli-5.5.0.zip  
  From: https://github.com/ontop/ontop/releases

- **Apache Jena Fuseki 6.2.0**  
  https://dlcdn.apache.org/jena/binaries/apache-jena-fuseki-6.2.0.zip  
  From: https://jena.apache.org/download/index.cgi

## Benchmark queries

The same three SPARQL queries are executed against each engine.

| Query | Description | File |
|---|---|---|
| Query 1 | Five molecules with redox potential | `queries/query_bm1.rq` |
| Query 2 | Five molecules with experimental redox potential at pH 0 | `queries/query_bm2.rq` |

## Notes

- RDFLib loads the materialized Turtle file once before benchmarking; graph-loading time is not included in query execution time.
- Endpoint benchmarks measure client-observed response time, including HTTP communication.
- Process RAM values represent the maximum resident memory used by the Python/Jupyter process for RDFLib, and by the Java endpoint process for Ontop and Fuseki.
- RDFLib process RAM includes the loaded RDF graph as well as Jupyter/Python overhead.
- Ontop and Fuseki process RAM includes JVM overhead, endpoint caches, and other server resources; it is not memory used by one query alone.
- Storage used represents the size of the data files used by each approach. It does not include the Java runtime, Fuseki/Ontop installation, or Python packages.
- Materialization time is the time required to create the materialized Turtle file used by RDFLib and Fuseki. Ontop does not require RDF materialization because it queries the H2 database through the OBDA mapping.
- Each query is run once as a warm-up, followed by multiple measured repetitions.

In [ ]:
from __future__ import annotations

import logging
import time
import tracemalloc
import warnings
from pathlib import Path
import psutil

import pandas as pd
import requests
from rdflib import Graph

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

# -------------------------------------------------------------------
# Paths and benchmark configuration
# -------------------------------------------------------------------

MATERIALIZED_TTL = Path("../../database/triples/OFBO-materialized.ttl")

OBDA_FILE = Path("../../ontology/OFBO.obda")
ONTOLOGY_FILE = Path("../../ontology/OFBO.ttl")

H2_DATABASE_FILES = [
    Path("../../database/h2/ofbo_db.mv.db"),
    Path("../../database/h2/ofbo_db.trace.db"),
]

QUERIES = {
    "Query 1": Path("queries/query_bm1.rq"),
    "Query 2": Path("queries/query_bm2.rq"),
}

FUSEKI_ENDPOINT = "http://localhost:3030/ofbo/query"
ONTOP_ENDPOINT = "http://localhost:8080/sparql"

ITERATIONS = 50
TIMEOUT_SECONDS = 120

## Start the SPARQL services

Start Fuseki and Ontop before running the benchmark.

Example commands (run from this folder):

```bash

### Fuseki

fuseki-server --file=../../database/triples/OFBO-materialized.ttl /ofbo

### Ontop

ontop endpoint -m ../../ontology/OFBO.obda -p ../../ontology/OFBO.properties -t ../../ontology/OFBO.ttl
```

In [2]:
# Load the RDFLib graph once
graph = Graph()
graph.parse(MATERIALIZED_TTL, format="turtle")

print(f"RDFLib graph loaded: {len(graph)} triples")


def file_size_mb(files):
    total_bytes = sum(
        file.stat().st_size
        for file in files
        if file.exists()
    )

    return total_bytes / (1024 * 1024)

# Query functions
def run_rdflib_query(query_path):
    with open(query_path, "r") as file:
        query = file.read()

    return list(graph.query(query))


def run_fuseki_query(query_path):
    with open(query_path, "r") as file:
        query = file.read()

    response = requests.get(
        FUSEKI_ENDPOINT,
        params={"query": query},
        headers={"Accept": "text/csv"},
        timeout=TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    return response.text


def run_ontop_query(query_path):
    with open(query_path, "r") as file:
        query = file.read()

    response = requests.get(
        ONTOP_ENDPOINT,
        params={"query": query},
        headers={"Accept": "text/csv"},
        timeout=TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    return response.text

RDFLib graph loaded: 315160 triples


In [6]:
materialization_time = {
    "RDFLib": 10.2,
    "Fuseki": 10.2,
    "Ontop": float("nan"),
}


storage_used = {
    "RDFLib": file_size_mb([MATERIALIZED_TTL]),
    "Fuseki": file_size_mb([MATERIALIZED_TTL]),
    "Ontop": file_size_mb([
        OBDA_FILE,
        ONTOLOGY_FILE,
        *H2_DATABASE_FILES,
    ]),
}


def endpoint_ram_mb(engine_name):
    for process in psutil.process_iter(["cmdline", "memory_info"]):
        try:
            command = " ".join(process.info["cmdline"] or []).lower()

            if engine_name == "Fuseki" and "fuseki" in command:
                return process.info["memory_info"].rss / (1024 * 1024)

            if engine_name == "Ontop" and "ontop" in command:
                return process.info["memory_info"].rss / (1024 * 1024)

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    return None


def process_ram_mb(engine_name):
    if engine_name == "RDFLib":
        return psutil.Process().memory_info().rss / (1024 * 1024)

    return endpoint_ram_mb(engine_name)


def benchmark(function, query_path, engine_name):
    # Warm-up run, not included in timing
    function(query_path)

    peak_ram = process_ram_mb(engine_name)

    start_time = time.perf_counter()

    for _ in range(ITERATIONS):
        function(query_path)

        current_ram = process_ram_mb(engine_name)

        if current_ram is not None:
            if peak_ram is None:
                peak_ram = current_ram
            else:
                peak_ram = max(peak_ram, current_ram)

    total_time = time.perf_counter() - start_time

    return {
        "Total time (s)": total_time,
        "Time/query (s)": total_time / ITERATIONS,
        "Process RAM (MB)": peak_ram,
    }


engines = {
    "RDFLib": run_rdflib_query,
    "Ontop": run_ontop_query,
    "Fuseki": run_fuseki_query,
}

results = []

for query_name, query_path in QUERIES.items():
    for engine_name, function in engines.items():
        result = benchmark(function, query_path, engine_name)

        results.append({
            "Query": query_name,
            "Engine": engine_name,
            "Iterations": ITERATIONS,
            **result,
            "Storage used (MB)": storage_used[engine_name],
            "Materialization time (s)": materialization_time[engine_name],
        })


results_df = pd.DataFrame(results)

results_df = results_df.round({
    "Total time (s)": 4,
    "Time/query (s)": 4,
    "Process RAM (MB)": 2,
    "Storage used (MB)": 2,
    "Materialization time (s)": 2,
})


for query_name in QUERIES:
    print(query_name)

    query_results = results_df[
        results_df["Query"] == query_name
    ].drop(columns="Query")

    display(query_results)

Query 1


,Engine,Iterations,Total time (s),Time/query (s),Process RAM (MB),Storage used (MB),Materialization time (s)
0,RDFLib,50,2.9899,0.0598,444.17,39.99,10.2
1,Ontop,50,1.5086,0.0302,573.65,0.77,NaN
2,Fuseki,50,1.7164,0.0343,2255.28,39.99,10.2


Query 2


,Engine,Iterations,Total time (s),Time/query (s),Process RAM (MB),Storage used (MB),Materialization time (s)
3,RDFLib,50,0.6261,0.0125,444.54,39.99,10.2
4,Ontop,50,1.4651,0.0293,573.79,0.77,NaN
5,Fuseki,50,1.5334,0.0307,2513.49,39.99,10.2
